# Lesson 8.1 — Capstone: Manipulability & Singularity Analyzer
**Module 6 · Unit 8 · Lesson 29**

`analyze(q)` returns the full capability report (σ, w, κ, ellipsoid axes, singularity flag) from **one SVD** — Units 4–6 in a single reusable diagnostic. Companion to the *Resolved-Rate Tracker* demo.

In [1]:
import numpy as np
def dh(th,d,a,al):
    ct,st,ca,sa=np.cos(th),np.sin(th),np.cos(al),np.sin(al)
    return np.array([[ct,-st*ca,st*sa,a*ct],[st,ct*ca,-ct*sa,a*st],[0,sa,ca,d],[0,0,0,1]])
def forward_chain(P,T,q):
    M=np.eye(4); Ms=[M.copy()]
    for i,(th0,d0,a,al) in enumerate(P):
        th,d=(th0+q[i],d0) if T[i]=="R" else (th0,d0+q[i]); M=M@dh(th,d,a,al); Ms.append(M.copy())
    return Ms
def geometric_jacobian(P,T,q):
    Ms=forward_chain(P,T,q); on=Ms[-1][:3,3]; J=np.zeros((6,len(q)))
    for i in range(len(q)):
        z=Ms[i][:3,2]; o=Ms[i][:3,3]
        if T[i]=="R": J[:3,i]=np.cross(z,on-o); J[3:,i]=z
        else: J[:3,i]=z
    return J
def Jv_planar(P,T,q): return geometric_jacobian(P,T,q)[:2,:]
def fk_xy(P,T,q): return forward_chain(P,T,q)[-1][:2,3]
P2=[(0,0,1,0),(0,0,1,0)]; T2=["R","R"]
P3=[(0,0,1,0),(0,0,1,0),(0,0,0.6,0)]; T3=["R","R","R"]   # redundant (2D task, 3 joints)
EPS=0.08   # singularity threshold on sigma_min


In [2]:
def analyze(P,T,q):
    """One SVD -> full capability report (Units 4-6 in one function)."""
    J=Jv_planar(P,T,q); U,S,Vt=np.linalg.svd(J)
    w=float(np.prod(S)); kappa=float(S[0]/max(S[-1],1e-12))
    return {"sigma":S,"w":w,"kappa":kappa,
            "axes":[(U[:,i],float(S[i])) for i in range(len(S))],
            "singular":bool(S[-1]<EPS),"sigma_min":float(S[-1]),"J":J,"U":U,"Vt":Vt}


## Healthy vs near-singular pose: the flag fires correctly

In [3]:
checks=[]
healthy=analyze(P2,T2,np.array([0.4,1.2]))
near=analyze(P2,T2,np.array([0.4,0.05]))
for name,r in [("healthy",healthy),("near-singular",near)]:
    print(f"{name:13s} sigma={np.round(r['sigma'],3)}  w={r['w']:.3f}  kappa={r['kappa']:.1f}  singular={r['singular']}")
checks.append(healthy["singular"]==False and near["singular"]==True)

healthy       sigma=[1.864 0.5  ]  w=0.932  kappa=3.7  singular=False
near-singular sigma=[2.235 0.022]  w=0.050  kappa=100.0  singular=True


## w and kappa are consistent with the singular values

In [4]:
r=analyze(P2,T2,np.array([0.3,0.9]))
checks.append(np.isclose(r["w"],np.prod(r["sigma"]),atol=1e-9))
checks.append(np.isclose(r["kappa"],r["sigma"][0]/r["sigma"][-1],atol=1e-9))
print("w==prod(sigma):",checks[-2],"  kappa==s1/sn:",checks[-1])
print("ellipsoid axes (u, length):",[(np.round(u,2),round(l,3)) for u,l in r["axes"]])
assert all(checks), f"FAILED: {checks}"
print("All checks passed.")

w==prod(sigma): True   kappa==s1/sn: True
ellipsoid axes (u, length): [(array([-0.75,  0.66]), 2.023), (array([0.66, 0.75]), 0.387)]
All checks passed.
